> **Ekip paylaşım kopyası:** Kayıtlı çıktılar ve widget durumu dosya boyutunu azaltmak için temizlenmiştir; analiz hücreleri korunur. Veri dosyası Git deposuna dahil değildir. Kurulum, veri yolu ve mevcut veri dosyasını kopyalamadan kullanma seçenekleri için `../README.md` dosyasına bakın. `.venv` kernel'ini seçip Run All çalıştırın. Kaynak projedeki notebook ve çıktılar değiştirilmemiştir.

# DS02 motor ve uçuş inceleyici — yalnızca dev
Motor ve uçuş seçin; **Uçuşu yükle** düğmesine basın. Tablo bütün 47 özgün değişkeni ve `sample_index` sütununu içerir. Uzun tablolar sayfalanır; CSV seçili uçuşun tamamını içerir.

`sample_index`, seçilen uçuş içinde sıfırdan başlayan örnek sırasıdır; **saniye veya zaman damgası değildir**. Dosyadaki artan satır sırası korunur; fiziksel zaman bilgisi olmadığı için gerçek zaman aralıkları çıkarılmaz.

Gerekli paketler: h5py, numpy, pandas, ipywidgets, ipykernel. VS Code'da proje `.venv` kernel'ini seçip bütün hücreleri çalıştırın. Önceki notebook değiştirilmez.

### Öğretici okuma rehberi ve değişken sözlüğü
[DS02 Türkçe değişken sözlüğü](../docs/DS02_Degisken_Sozlugu.md), 47 özgün değişkenin tam İngilizce/Türkçe adlarını, birimlerini, bileşenlerini ve yorum sınırlarını açıklar. A kimlik/etiket, W koşul, X_s fiziksel ölçülebilir yanıt, X_v sanal model çıktısı, T simülasyon iç sağlığı, Y/RUL hedeftir. Fiziksel sensör adı bütün yanıtların gerçek bir arızalı motordan kaydedildiği anlamına gelmez; N-CMAPSS motor yanıtları simüle edilmiştir.

Makale Tables 4–8'e göre sıcaklık °R, basınç psia, hız rpm, kütlesel debi pps/lbm/s'dir. TRA birimi %; stall marjları kaynakta boyutsuz gösterilir. Grafik kodu değiştirilmediği için “özgün değer” etiketlerinin fiziksel birimleri sözlükten okunmalıdır. Önceki birim-doğrulama notlarının üzerine bu yeni kaynak açıklaması eklenmiştir; hesaplamalar değişmemiştir.


In [ ]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Paylaşım kopyası: veri yolu bilgisayardan bağımsız çözülür.
import os

def resolve_ncmapss_paths(start=None):
    working_directory = Path.cwd().resolve() if start is None else Path(start).resolve()
    project_root = None
    for parent in [working_directory, *working_directory.parents]:
        if parent.name == "n_cmapss" and (parent / "notebooks").is_dir():
            project_root = parent
            break
        candidate = parent / "n_cmapss"
        if (candidate / "notebooks").is_dir():
            project_root = candidate
            break
    if project_root is None:
        raise FileNotFoundError("Notebook'u ortak repo, n_cmapss veya n_cmapss/notebooks klasöründen çalıştırın.")
    external_path = os.environ.get("N_CMAPSS_H5")
    h5_path = Path(external_path).expanduser().resolve() if external_path else project_root / "data" / "N-CMAPSS_DS02-006.h5"
    return project_root, h5_path

project_root, h5_path = resolve_ncmapss_paths()
if not h5_path.is_file():
    raise FileNotFoundError(
        f"Veri dosyası bulunamadı: {h5_path}. HDF5 repoya dahil değildir. "
        "Dosyayı yerel n_cmapss/data/ altında sağlayın veya kernel ortamında "
        "N_CMAPSS_H5 değişkenini mevcut dosyanın tam yoluna ayarlayın. README'ye bakın."
    )

DEV_KEYS = ["A_dev", "W_dev", "X_s_dev", "X_v_dev", "T_dev", "Y_dev"]
VAR_KEYS = ["A_var", "W_var", "X_s_var", "X_v_var", "T_var"]
CHUNK_ROWS = 100_000

def decode_names(dataset):
    return [v.decode("utf-8").rstrip("\x00") if isinstance(v, (bytes, np.bytes_)) else str(v)
            for v in np.asarray(dataset[()]).reshape(-1)]

with h5py.File(h5_path, "r") as h5:
    names = {key: decode_names(h5[key]) for key in VAR_KEYS}
    column_names = {key: names[var] for key, var in zip(DEV_KEYS[:-1], VAR_KEYS)}
    column_names["Y_dev"] = ["RUL"]
    row_counts = {key: h5[key].shape[0] for key in DEV_KEYS}
    if len(set(row_counts.values())) != 1:
        raise ValueError(f"Satır sayıları eşit değil: {row_counts}")
    for key in DEV_KEYS:
        if h5[key].ndim != 2 or h5[key].shape[1] != len(column_names[key]):
            raise ValueError(f"Sütun metadata uyumsuzluğu: {key}")
all_columns = [name for key in DEV_KEYS for name in column_names[key]]
assert len(all_columns) == len(set(all_columns)) == 47
unit_col = names["A_var"].index("unit")
cycle_col = names["A_var"].index("cycle")
print("dev satır sayısı:", row_counts["A_dev"])


In [ ]:
# Grafik etiketleri için tek merkezi başvuru; sayısal veriye dönüşüm uygulanmaz.
import json as _ds02_json
from textwrap import fill as _ds02_wrap
_ds02_metadata_path = project_root / "docs" / "assets" / "ds02_variable_metadata.json"
with _ds02_metadata_path.open(encoding="utf-8") as _ds02_stream:
    _ds02_metadata = _ds02_json.load(_ds02_stream)
DS02_VARIABLES = _ds02_metadata["variables"]
AXIS_LABELS = _ds02_metadata["axis_labels"]

def variable_unit(name):
    return DS02_VARIABLES[name]["compact_unit"]

def variable_title(name):
    return f"{name} ({variable_unit(name)})"

def variable_label(name):
    entry = DS02_VARIABLES[name]
    return _ds02_wrap(f"{name} — {entry['turkish']} ({entry['unit']})", width=70)


> [Motor ve Sensör Atlası](../docs/DS02_Motor_ve_Sensor_Atlasi.md) ve [merkezi birim sözlüğü](../docs/assets/ds02_variable_metadata.json) bu notebook'un etiket başvurusudur. Hesaplama/veri yolu mantığı korunmuştur.


## Motor/uçuş kataloğu: sınırlı bellekle tarama
`A_dev` içinden yalnızca unit ve cycle sütunları en fazla 100.000 satırlık parçalarla okunur. Her motor/uçuş için eşleşen satırların yarı açık `[başlangıç, bitiş)` aralıkları saklanır. Parça sınırından geçen bitişik aralıklar birleştirilir; aynı uçuşun dosyada birden fazla yerde bulunması da desteklenir. Milyonlarca satırlık bir pandas tablosu veya tam satır indeksi dizisi oluşturulmaz.

### Motor, cycle ve uçuş sınıfı
unit=2 motor kimliği, cycle=1 ilk uçuş, Fc=3 uzun görev sınıfıdır. DS02 dev'deki altı motorun Fc=3 olması bu bölümün kapsamıdır; bütün N-CMAPSS'in tek sınıflı olduğu anlamına gelmez. Sınıf uçuşun bütün profilinden atanır; kayıt satırlarından saat türetilmez. [Fc ayrıntıları](../docs/DS02_Degisken_Sozlugu.md#fc-siniflari).

In [ ]:
flight_ranges = {}
with h5py.File(h5_path, "r") as h5:
    a = h5["A_dev"]
    for start in range(0, a.shape[0], CHUNK_ROWS):
        stop = min(start + CHUNK_ROWS, a.shape[0])
        units = a[start:stop, unit_col]
        cycles = a[start:stop, cycle_col]
        if not (np.isfinite(units).all() and np.isfinite(cycles).all()
                and np.equal(units, np.floor(units)).all()
                and np.equal(cycles, np.floor(cycles)).all()):
            raise ValueError("Motor/uçuş kimlikleri sonlu tam sayı olmalıdır")
        changes = np.flatnonzero((units[1:] != units[:-1]) | (cycles[1:] != cycles[:-1])) + 1
        boundaries = np.r_[0, changes, len(units)]
        for left, right in zip(boundaries[:-1], boundaries[1:]):
            key = (int(units[left]), int(cycles[left]))
            begin, end = start + int(left), start + int(right)
            ranges = flight_ranges.setdefault(key, [])
            if ranges and ranges[-1][1] == begin:
                ranges[-1] = (ranges[-1][0], end)
            else:
                ranges.append((begin, end))
engine_flights = {}
for unit, cycle in sorted(flight_ranges):
    engine_flights.setdefault(unit, []).append(cycle)
assert sum(end - begin for ranges in flight_ranges.values() for begin, end in ranges) == row_counts["A_dev"]
catalog = pd.DataFrame([
    {"unit": unit, "uçuş_sayısı": len(cycles), "ilk_cycle": min(cycles), "son_cycle": max(cycles)}
    for unit, cycles in engine_flights.items()
])
display(catalog)
print("Toplam motor/uçuş çifti:", len(flight_ranges))


## Seçili uçuşu birleştirme ve CSV
Yalnızca seçilen uçuşun aralıkları her dev datasetinden aynı sırayla okunur. Bellek kullanımı tek uçuşun boyutuyla sınırlıdır. Birleştirme satır indeksiyle yapılır. CSV dosyaları `outputs/csv/` altında oluşturulur; mevcut dosyanın üzerine yazılmaz.

### Motor atlası ve grafik/tablo birimleri
[Motor ve Sensör Atlası](../docs/DS02_Motor_ve_Sensor_Atlasi.md) bölge eşleştirmelerini, özgün kesit şemasını ve artış/azalışta araştırılabilecek olasılıkları açıklar. [47 değişken sözlüğü](../docs/DS02_Degisken_Sozlugu.md) korunmuştur. Birimler [merkezi JSON etiket sözlüğünden](../docs/assets/ds02_variable_metadata.json) alınır; veri dönüştürülmez, özgün sütun adları değişmez.

**TRA: % (Table 4), derece değil.** phi küçük başlıklarda pps/psi* ile gösterilir: yıldız kaynak biriminin korunduğunu ama yerel ölçeğin doğrulanmadığını belirtir. Stall margin ve sağlık parametreleri boyutsuzdur; yüzde sağlık/verim kaybı değildir. Medyan/Q1–Q3 ve min–maksimum bantlarının önceki anlamları korunur. sample_index ölçüm sırası (saniye değil), cycle uçuş döngüsü, RUL kalan uçuş döngüsüdür. Normalize yaşam 0–1, boyutsuz ve yalnızca görsel karşılaştırmadır.

<details><summary>Değişken adları, fiziksel anlamları ve birimleri</summary>

| Kod | Türkçe açıklama | Birim |
|---|---|---|
| `unit` | Motor kimliği | etiket |
| `cycle` | Uçuş döngüsü numarası | uçuş döngüsü |
| `Fc` | Uçuş sınıfı | etiket |
| `hs` | Sağlık durumu etiketi | etiket |
| `alt` | Uçuş irtifası | ft |
| `Mach` | Uçuş Mach sayısı | boyutsuz |
| `TRA` | Gaz kolu / güç talebi göstergesi | % |
| `T2` | Fan giriş toplam sıcaklığı | °R |
| `T24` | LPC çıkış toplam sıcaklığı | °R |
| `T30` | HPC çıkış toplam sıcaklığı | °R |
| `T48` | HPT çıkış toplam sıcaklığı | °R |
| `T50` | LPT çıkış toplam sıcaklığı | °R |
| `P15` | Bypass kanalı toplam basıncı | psia |
| `P2` | Fan giriş toplam basıncı | psia |
| `P21` | Fan çıkış toplam basıncı | psia |
| `P24` | LPC çıkış toplam basıncı | psia |
| `Ps30` | HPC çıkış statik basıncı | psia |
| `P40` | Yanma odası çıkışı / HPT girişi toplam basıncı | psia |
| `P50` | LPT çıkış toplam basıncı | psia |
| `Nf` | Fanın fiziksel dönüş hızı | rpm |
| `Nc` | Çekirdeğin fiziksel dönüş hızı | rpm |
| `Wf` | Yakıt kütlesel debisi | lbm/s |
| `T40` | Yanma odası çıkış toplam sıcaklığı | °R |
| `P30` | HPC çıkış toplam basıncı | psia |
| `P45` | HPT çıkış toplam basıncı | psia |
| `W21` | Fan kütlesel debisi | lbm/s |
| `W22` | LPC çıkış kütlesel debisi | lbm/s |
| `W25` | HPC giriş kütlesel debisi | lbm/s |
| `W31` | HPT soğutma bleed debisi, W31 | lbm/s |
| `W32` | HPT soğutma bleed debisi, W32 | lbm/s |
| `W48` | HPT çıkış kütlesel debisi | lbm/s |
| `W50` | LPT çıkış kütlesel debisi | lbm/s |
| `SmFan` | Fan stall marjı | boyutsuz |
| `SmLPC` | LPC stall marjı | boyutsuz |
| `SmHPC` | HPC stall marjı | boyutsuz |
| `phi` | Yakıt debisinin Ps30'a oranı (makale tanımı) | pps/psi; dosya ölçeği belirsiz |
| `fan_eff_mod` | Fan verim değiştiricisi | boyutsuz |
| `fan_flow_mod` | Fan akış kapasitesi değiştiricisi | boyutsuz |
| `LPC_eff_mod` | LPC verim değiştiricisi | boyutsuz |
| `LPC_flow_mod` | LPC akış kapasitesi değiştiricisi | boyutsuz |
| `HPC_eff_mod` | HPC verim değiştiricisi | boyutsuz |
| `HPC_flow_mod` | HPC akış kapasitesi değiştiricisi | boyutsuz |
| `HPT_eff_mod` | HPT verim değiştiricisi | boyutsuz |
| `HPT_flow_mod` | HPT akış kapasitesi değiştiricisi | boyutsuz |
| `LPT_eff_mod` | LPT verim değiştiricisi | boyutsuz |
| `LPT_flow_mod` | LPT akış kapasitesi değiştiricisi | boyutsuz |
| `RUL` | Kalan faydalı ömür | uçuş döngüsü |
| `sample_index` | Uçuş içindeki ölçüm sırası | örnek sırası; saniye değil |

</details>

**Kayıtlı çıktı notu:** Kaynak notebook'lardaki başlangıç grafiklerinin görselleri, çizgi/bant verilerinin eşitliği doğrulanarak mevcut CSV özetleri ve tek uçuş verisiyle yeni etiketlerle yeniden çizildi. Veri içeriği değiştirilmedi. Paylaşım kopyalarının temiz çıktı durumu korunur. Tam EDA yeniden çalıştırılmadı.


### 48 sütunlu tablonun okuma anahtarı
47 özgün sütuna sample_index eklenir: bu uçuş içi örnek sırasıdır, saniye değildir. A'daki unit/cycle/Fc/hs sensör değildir; W koşul, X_s/X_v yanıt, T iç sağlık, Y hedef olarak okunur. Ps30 statik, P30 toplam basınçtır; Wf yakıt, W21 fan kütlesel akışıdır. [X_s](../docs/DS02_Degisken_Sozlugu.md#xs-fiziksel), [X_v](../docs/DS02_Degisken_Sozlugu.md#xv-sanal), [türetilmiş alanlar](../docs/DS02_Degisken_Sozlugu.md#turetilmis).

T parametreleri sensör, debi veya doğrudan yüzde verim kaybı değildir. phi'nin makale tanımı yalın Wf/Ps30 hesabıyla yerel ilk satırda uyuşmaz; çözülmemiş ölçek sorusu sözlükte açıklanır. CSV'nin bütün 48 sütunu içermesi model girdilerinin seçildiği anlamına gelmez.

In [ ]:
def load_flight(unit, cycle):
    key = (int(unit), int(cycle))
    if key not in flight_ranges:
        raise ValueError(f"Motor/uçuş bulunamadı: {key}")
    ranges = flight_ranges[key]
    frames = []
    with h5py.File(h5_path, "r") as h5:
        for name in DEV_KEYS:
            blocks = [h5[name][begin:end, :] for begin, end in ranges]
            values = blocks[0] if len(blocks) == 1 else np.concatenate(blocks, axis=0)
            frames.append(pd.DataFrame(values, columns=column_names[name]))
    table = pd.concat(frames, axis=1, verify_integrity=True)
    table.insert(0, "sample_index", np.arange(len(table), dtype=np.int64))
    if not ((table["unit"] == key[0]).all() and (table["cycle"] == key[1]).all()):
        raise ValueError("Motor/uçuş filtreleme hatası")
    return table

def export_csv(table, unit, cycle):
    directory = project_root / "outputs" / "csv"
    directory.mkdir(parents=True, exist_ok=True)
    stem = f"DS02_dev_unit_{int(unit)}_cycle_{int(cycle)}"
    suffix = 0
    while True:
        path = directory / (stem + (f"_{suffix}" if suffix else "") + ".csv")
        try:
            with path.open("x", encoding="utf-8-sig", newline="") as stream:
                table.to_csv(stream, index=False)
            return path
        except FileExistsError:
            suffix += 1


def scrollable_table_html(table):
    # 400 px yükseklik, alt yatay kaydırma çubuğunu görünür alan içinde tutar.
    # overflow:scroll iki eksendeki kaydırma alanını sürekli ayırır.
    html = table.to_html(index=False, border=0, escape=True, max_rows=None, max_cols=None)
    return ("<div role='region' aria-label='Uçuş verileri: yatay ve dikey kaydırılabilir tablo' "
            "tabindex='0' style='width:100%;height:400px;overflow-x:scroll;overflow-y:scroll;"
            "scrollbar-gutter:stable;border:1px solid #888;box-sizing:border-box;'>"
            + html.replace('<table ', "<table style='width:max-content;min-width:100%;white-space:nowrap;' ")
            + "</div>")


## Doğrulama: motor 2, uçuş 1
Satır sayısı katalogdaki aralıklara karşı kontrol edilir. Bütün 47 sütun, aynı HDF5 aralıklarıyla parça parça karşılaştırılır. Örnek sırası ve RUL doğrulanır. CSV gidiş/dönüş testi geçici klasörde yapılır; kalıcı CSV yalnızca kullanıcı düğmeye bastığında üretilir.

In [ ]:
test_table = load_flight(2, 1)
test_ranges = flight_ranges[(2, 1)]
expected_rows = sum(end - begin for begin, end in test_ranges)
assert len(test_table) == expected_rows
assert test_table.shape[1] == 48  # 47 özgün değişken + sample_index
assert np.array_equal(test_table["sample_index"], np.arange(expected_rows))
with h5py.File(h5_path, "r") as h5:
    offset = 0
    for begin, end in test_ranges:
        for pos in range(begin, end, CHUNK_ROWS):
            stop = min(pos + CHUNK_ROWS, end)
            n = stop - pos
            for key in DEV_KEYS:
                np.testing.assert_array_equal(
                    test_table.iloc[offset:offset+n][column_names[key]].to_numpy(),
                    h5[key][pos:stop, :])
            offset += n
    expected_rul = h5["Y_dev"][test_ranges[0][0], 0]
assert test_table.iloc[0]["RUL"] == expected_rul == 74
import tempfile
with tempfile.TemporaryDirectory() as temporary:
    csv_path = Path(temporary) / "flight_test.csv"
    test_table.to_csv(csv_path, index=False)
    restored = pd.read_csv(csv_path)
    pd.testing.assert_frame_equal(test_table, restored, check_exact=False, rtol=1e-12, atol=1e-12)
print(f"Motor 2 / uçuş 1: {len(test_table):,} satır, {test_table.shape[1]} sütun")
print(f"İlk satır RUL: {int(expected_rul)}")
print("Bütün değişkenler, örnek sırası ve CSV gidiş/dönüş kontrolü başarılı.")
display(widgets.HTML(value=scrollable_table_html(test_table.head(10)),
                     layout=widgets.Layout(width="950px", max_width="100%")))


## Etkileşimli inceleme
Motor değişince uçuş seçenekleri güncellenir. **Uçuşu yükle** seçimi belleğe alır. Her sayfada 50 örnek ve bütün sütunlar gösterilir; yatay kaydırma gerekebilir. CSV düğmesi en son yüklenen uçuşun bütün satırlarını aktarır. Seçimler değiştiğinde eski tablo ve dışa aktarma devre dışı kalır. İlk yüklenen seçim motor 2 / uçuş 1'dir.

**Sütun grubu** yalnızca ekrandaki sütunları seçer; `sample_index`, `unit`, `cycle` her grupta korunur. CSV bütün 48 sütunu içerir. Tablo alanı 950 px genişlikte (dar ekranda %100) ve 400 px yüksekliktedir; yatay kaydırma alanı bu kutunun altındadır. İşletim sisteminin otomatik gizlenen çubuk ayarı çubuğun görünümünü etkileyebilir; kutuya odaklanıp Shift + fare tekerleği veya dokunmatik yüzey ile de yatay kaydırılabilir.


### Sütun gruplarını birlikte okuma
İrtifa veya güç komutu değişirken T24/Nf değişimi tek başına bozulma kanıtı değildir. hs/T ground-truth veya model iç bilgisi, RUL hedef içerir. Gerçekçi model için bunların erişilebilirlik/sızıntı sınırları ayrıca değerlendirilmelidir. [Model girdisi sınırları](../docs/DS02_Degisken_Sozlugu.md#model-sinirlari).

In [ ]:
engine = widgets.Dropdown(options=list(engine_flights), value=2, description="Motor:")
flight = widgets.Dropdown(options=engine_flights[2], value=1, description="Uçuş:")
load_button = widgets.Button(description="Uçuşu yükle", button_style="primary")
csv_button = widgets.Button(description="CSV dışa aktar")
page = widgets.BoundedIntText(value=1, min=1, max=max(1, (len(test_table)+49)//50), description="Sayfa:")
status = widgets.HTML()
# HTML widget'in dış genişliği sabittir; dar çıktılarda kullanılabilir alana sığar.
# İç alanın yüksekliği sınırlıdır: yatay çubuk uzun tablonun altında kaybolmaz.
view = widgets.HTML(layout=widgets.Layout(width="950px", max_width="100%"))
identity_columns = ["sample_index", "unit", "cycle"]
column_groups = {
    "Tümü (48 sütun)": list(test_table.columns),
    "Uçuş bilgileri ve koşulları (A + W)": column_names["A_dev"] + column_names["W_dev"],
    "Fiziksel sensörler (X_s)": column_names["X_s_dev"],
    "Sanal sensörler (X_v)": column_names["X_v_dev"],
    "Bozulma parametreleri (T)": column_names["T_dev"],
    "RUL": ["RUL"],
}
column_group = widgets.Dropdown(options=list(column_groups), description="Sütun grubu:",
    style={"description_width": "100px"}, layout=widgets.Layout(width="480px", max_width="100%"))

def visible_columns():
    # Kimlik sütunları her grupta bir kez ve solda gösterilir.
    return list(dict.fromkeys(identity_columns + column_groups[column_group.value]))
message = widgets.Output()
state = {"table": test_table, "key": (2, 1)}

def show_page(change=None):
    table = state["table"]
    if table is None:
        view.value = "<p>Seçilen uçuşu yüklemek için düğmeye basın.</p>"
        return
    start = (page.value - 1) * 50
    columns = visible_columns()
    status.value = (f"Motor {state['key'][0]} / uçuş {state['key'][1]} — "
                    f"{len(table):,} satır; ekranda {len(columns)}/48 sütun; "
                    f"sayfa {page.value}/{page.max}. "
                    "sample_index: örnek sırası (saniye değil). CSV: bütün 48 sütun.")
    # Görünüm filtresi state içindeki tam uçuş tablosunu değiştirmez.
    view.value = scrollable_table_html(table.iloc[start:start+50][columns])

def invalidate():
    state.update(table=None, key=None)
    csv_button.disabled = True
    status.value = "Yeni seçim henüz yüklenmedi."
    with message:
        clear_output()
    show_page()

def change_engine(change):
    flight.options = engine_flights[change["new"]]
    flight.value = engine_flights[change["new"]][0]
    invalidate()

def change_flight(change):
    invalidate()

def on_load(button):
    load_button.disabled = True
    csv_button.disabled = True
    try:
        table = load_flight(engine.value, flight.value)
        state.update(table=table, key=(engine.value, flight.value))
        page.max = max(1, (len(table)+49)//50)
        page.value = 1
        csv_button.disabled = False
        show_page()
    except Exception as exc:
        invalidate()
        with message:
            print(f"Yükleme hatası: {exc}")
    finally:
        load_button.disabled = False

def on_export(button):
    with message:
        clear_output()
        try:
            if state["table"] is None:
                raise ValueError("Önce bir uçuş yükleyin.")
            path = export_csv(state["table"], *state["key"])
            print(f"CSV kaydedildi: {path}")
        except Exception as exc:
            print(f"CSV hatası: {exc}")

engine.observe(change_engine, names="value")
flight.observe(change_flight, names="value")
page.observe(show_page, names="value")
column_group.observe(show_page, names="value")
load_button.on_click(on_load)
csv_button.on_click(on_export)
display(widgets.VBox([widgets.HBox([engine, flight]), widgets.HBox([load_button, csv_button, page]), column_group, status, view, message]))
show_page()
